# Photo → lively avatar clip (LivePortrait)

Turns one photo into a ~27 s clip with natural head nods, tilts, blinks and expression changes,
by chaining three talking-style LivePortrait driving clips (d20, d19, d13). The clip then becomes
the live agent's reference video: MuseTalk lip-syncs it in real time (worker notebook,
`AVATAR_VIDEO`). Only the head and face move — a photo has no hands to animate.

**Run this in its own session**, not in the worker notebook (LivePortrait next to the running
worker/MuseTalk ended a session before). Settings: **GPU T4 x2**, **Internet On**, and attach the
dataset with your photo (**Add Input**). Front-facing, sharp, unobstructed face works best. Only
use photos you have the rights and consent to animate.

In [ ]:
import glob, os, subprocess
PHOTO = ""   # leave empty to use the first image found under /kaggle/input
if not PHOTO:
    images = sorted(f for f in glob.glob("/kaggle/input/**/*", recursive=True)
                    if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp")))
    assert images, "no image under /kaggle/input -- attach your dataset with Add Input"
    PHOTO = images[0]
    print("found:", images)
print("using photo:", PHOTO)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# LivePortrait + the packages it imports (its pinned requirements don't exist for Python 3.13;
# CPU onnxruntime only does one-off face detection -- onnxruntime-gpu is deliberately not used).
# Ends by listing the weight files; if the session restarts during this cell, tell us at which step.
LP = "/kaggle/working/LivePortrait"
if not os.path.exists(f"{LP}/pretrained_weights/liveportrait"):
    print("1/3 cloning LivePortrait"); !rm -rf {LP} && git clone -q --depth 1 https://github.com/KwaiVGI/LivePortrait.git {LP}
    print("2/3 installing packages"); !pip install -q tyro pykalman rich lmdb albumentations onnx onnxruntime imageio-ffmpeg ffmpeg-python scikit-image requests
    print("3/3 downloading weights (629 MB)")
    from huggingface_hub import snapshot_download
    snapshot_download("KwaiVGI/LivePortrait", local_dir=f"{LP}/pretrained_weights",
                      ignore_patterns=["*.git*", "README.md", "docs/*", "*animal*",
                                       "liveportrait_animals/*"])
!ls {LP}/pretrained_weights/liveportrait

In [ ]:
# Render in the background, logging to files that survive a kernel restart: progress in
# render.log, memory every 2 s in mem.log. Then run the next cell to watch it.
import glob, os, subprocess, sys, urllib.request
LP = "/kaggle/working/LivePortrait"
assert os.path.exists(f"{LP}/pretrained_weights/liveportrait"), \
    "LivePortrait isn't set up in this session -- run the setup cell above first"
if "PHOTO" not in globals() or not PHOTO:
    PHOTO = sorted(f for f in glob.glob("/kaggle/input/**/*", recursive=True)
                   if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp")))[0]
import cv2
print("photo:", PHOTO, cv2.imread(PHOTO).shape)
for d in ("d20", "d19", "d13"):
    c = cv2.VideoCapture(f"{LP}/assets/examples/driving/{d}.mp4")
    print(f"driver {d}: {int(c.get(3))}x{int(c.get(4))}, {int(c.get(7))} frames")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/rooman-itsd/Rooman-Agent/feature/realtime-voice-loop/"
    "kaggle/musetalk_patches/photo_to_clip.py", "/kaggle/working/photo_to_clip.py")
OUT = "/kaggle/working/avatar-clip.mp4"
for f in ("render.log", "mem.log"):
    open(f"/kaggle/working/{f}", "w").close()
subprocess.Popen("while true; do free -m | awk 'NR==2{print strftime(\"%T\"), $3\" MB used, \"$7\" MB free\"}' "
                 ">> /kaggle/working/mem.log; sleep 2; done", shell=True)
subprocess.Popen(f'{sys.executable} -u /kaggle/working/photo_to_clip.py --photo "{PHOTO}" --lp-dir {LP} '
                 f'--out {OUT} --drivers d20,d19,d13 --multiplier 1.0 > /kaggle/working/render.log 2>&1',
                 shell=True)
print("rendering in the background -- run the next cell every minute or so")

In [ ]:
# Progress check (works even after a kernel restart). Done when it prints "CLIP ...".
!echo "--- render.log"; tail -12 /kaggle/working/render.log; echo "--- memory (last readings)"; tail -4 /kaggle/working/mem.log
!ls -la /kaggle/working/avatar-clip.mp4 2>/dev/null || echo "(clip not ready yet)"

In [ ]:
# Preview: 6 frames across the clip, then a download link.
import os, cv2, matplotlib.pyplot as plt
from IPython.display import FileLink, display
OUT = "/kaggle/working/avatar-clip.mp4"
assert os.path.exists(OUT), "clip not ready yet -- check progress with the cell above"
cap = cv2.VideoCapture(OUT); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fig, axes = plt.subplots(1, 6, figsize=(18, 4))
for ax, k in zip(axes, range(6)):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(k * (n - 1) / 5)); ok, f = cap.read()
    ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)); ax.axis("off"); ax.set_title(f"frame {int(k*(n-1)/5)}")
plt.show()
os.chdir("/kaggle/working"); display(FileLink("avatar-clip.mp4"))

**Next:** download `avatar-clip.mp4` (link above, or the Output panel → `/kaggle/working`), add it
to your dataset (dataset page → **New Version** → upload), then in the **worker** notebook set:

```python
AVATAR_VIDEO = "/kaggle/input/<your-dataset>/avatar-clip.mp4"
AVATAR_ID = "photo-lively"     # new id -> MuseTalk prepares the new clip
PHOTO = ""
```